<a href="https://colab.research.google.com/github/prithwis/fintech/blob/main/MNA_FreshMomentum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FreshMomentum — Monday Night Analysis (MNA)

**Output:** Three eligible candidates for discretionary selection (up to two purchases), with indicative limits 0.5% above the latest NSE close, not live LTP.\n\n**Inputs:** 12 daily NSE CM-UDiFF bhavcopies packaged by `Get-NSE12.ps1`; historical prices for the top 10 are fetched automatically from Yahoo Finance in Colab.

**Screen:** latest 10 sessions, EQ only, positive OLS slope of log closing price, R² > 0.80, average daily traded value ≥ ₹1 crore. Candidates ranked by slope.

**Last-day crack filter:** remove any of the original top 10 whose latest NSE close is below its preceding session close. Do not replenish the original top 10 from lower ranks.

**Extended-trend filter:** examine 40 historical sessions for the original top 10; exclude stocks with positive log-price slope and R² > 0.80 across the latest 35 sessions (approximately seven weeks). This is a reproducible proxy for extended momentum, **not a measured trend start date**. Failed history checks are excluded from BUY.

**SELL/HOLD:** placeholder only; tradebook integration is not yet implemented. **Do not use for live portfolio management while PM positions are open.**

**BUY:** at most two candidates from the verified top 10, excluding non-PM symbols; limit 0.5% above Monday close, rounded to tick, quantity sized at limit. These are indicative orders, not executions.


In [1]:
# CELL 1 — Configuration and input: upload the 12-session NSE archive once.
# WHY: keep market-data acquisition separate from the Colab screening workflow.
import io, re, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from google.colab import files

WINDOW = 10
MIN_R2 = 0.80
MIN_AVG_TRADED_VALUE = 1_00_00_000  # rupees per session
WEEKLY_BUDGET = 10_000             # edit as needed
MAX_POSITION_PCT = 0.50           # per-stock illustrative cap; user may choose up to two
SHORTLIST_SIZE = 3               # show three candidates; final choice is discretionary
BUY_PREMIUM = 0.005              # 0.5% above last NSE closing price (not live LTP)
TICK_SIZE = 0.05                 # verify actual NSE tick size for each instrument
TREND_LOOKBACK = 40             # historical sessions requested for top 10
EXTENDED_TREND_SESSIONS = 35    # seven trading weeks, approximately
TREND_MIN_R2 = 0.80             # operational proxy for a sustained upswing
EXCLUDE_SYMBOLS = {"AZAD", "PARAS"}  # non-PureMomentum positions
MAX_ABS_DAILY_MOVE = 0.40         # flag extreme close-to-close jumps (possible corporate actions)

print('Upload NSE12_YYYYMMDD.zip (the outer ZIP):')
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one NSE12 ZIP file.')
outer_name, outer_bytes = next(iter(uploaded.items()))
print(f'Uploaded: {outer_name} ({len(outer_bytes):,} bytes)')

Upload NSE12_YYYYMMDD.zip (the outer ZIP):


Saving NSE12_20261109_Input-Bhavcopy.zip to NSE12_20261109_Input-Bhavcopy.zip
Uploaded: NSE12_20261109_Input-Bhavcopy.zip (2,487,356 bytes)


In [2]:
# CELL 2 — Validate nested archives and assemble daily NSE observations.
# WHY: reject incomplete, duplicated or malformed source data before analysis.
# Validate all nested ZIPs and parse their NSE CM-UDiFF CSVs.
filename_re = re.compile(r'^BhavCopy_NSE_CM_0_0_0_(\d{8})_F_0000\.csv\.zip$', re.I)
required = {'TradDt','ISIN','TckrSymb','SctySrs','ClsPric','PrvsClsgPric','TtlTrfVal','TtlTradgVol'}
frames, audit = [], []
with zipfile.ZipFile(io.BytesIO(outer_bytes)) as outer:
    nested = [i for i in outer.infolist() if not i.is_dir()]
    if len(nested) != 12:
        raise ValueError(f'Expected 12 daily ZIPs, found {len(nested)} files in outer archive.')
    for member in nested:
        name = Path(member.filename).name
        m = filename_re.fullmatch(name)
        if not m:
            raise ValueError(f'Unexpected file in outer archive: {member.filename}')
        expected = pd.to_datetime(m.group(1), format='%Y%m%d')
        with zipfile.ZipFile(io.BytesIO(outer.read(member))) as inner:
            csvs = [i for i in inner.infolist() if not i.is_dir() and i.filename.lower().endswith('.csv')]
            if len(csvs) != 1:
                raise ValueError(f'{name}: expected exactly one CSV, found {len(csvs)}')
            with inner.open(csvs[0]) as f:
                df = pd.read_csv(f, low_memory=False)
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f'{name}: missing columns {sorted(missing)}')
        df['date'] = pd.to_datetime(df['TradDt'], errors='coerce').dt.normalize()
        if df.empty or df['date'].isna().any() or df['date'].nunique() != 1 or df['date'].iloc[0] != expected:
            raise ValueError(f'{name}: CSV date mismatch or invalid dates')
        if expected > pd.Timestamp.now(tz='Asia/Kolkata').tz_localize(None).normalize():
            raise ValueError(f'{name}: future session date')
        audit.append({'date': expected, 'filename': name, 'all_rows': len(df),
                      'eq_rows': int(df['SctySrs'].eq('EQ').sum())})
        frames.append(df)

session_audit = pd.DataFrame(audit).sort_values('date').reset_index(drop=True)
if session_audit['date'].duplicated().any():
    raise ValueError('Duplicate session dates in archive')
if len(session_audit) != 12:
    raise ValueError('Expected 12 distinct trading sessions')
if (session_audit['eq_rows'] < 100).any():
    raise ValueError('Unexpectedly few EQ rows in a session; inspect archive')
print('12 distinct sessions validated; latest:', session_audit.date.max().date())
display(session_audit)
raw = pd.concat(frames, ignore_index=True)


12 distinct sessions validated; latest: 2026-11-09


,date,filename,all_rows,eq_rows
0,2026-10-23,BhavCopy_NSE_CM_0_0_0_20261023_F_0000.csv.zip,3730,2706
1,2026-10-26,BhavCopy_NSE_CM_0_0_0_20261026_F_0000.csv.zip,3686,2709
2,2026-10-27,BhavCopy_NSE_CM_0_0_0_20261027_F_0000.csv.zip,3670,2712
3,2026-10-28,BhavCopy_NSE_CM_0_0_0_20261028_F_0000.csv.zip,3682,2711
4,2026-10-29,BhavCopy_NSE_CM_0_0_0_20261029_F_0000.csv.zip,3697,2712
5,2026-10-30,BhavCopy_NSE_CM_0_0_0_20261030_F_0000.csv.zip,3641,2703
6,2026-11-02,BhavCopy_NSE_CM_0_0_0_20261102_F_0000.csv.zip,3715,2714
7,2026-11-03,BhavCopy_NSE_CM_0_0_0_20261103_F_0000.csv.zip,3717,2744
8,2026-11-04,BhavCopy_NSE_CM_0_0_0_20261104_F_0000.csv.zip,3705,2739
9,2026-11-05,BhavCopy_NSE_CM_0_0_0_20261105_F_0000.csv.zip,3726,2742


In [3]:
# CELL 3 — Build the clean 10-session EQ price panel.
# WHY: require comparable, continuous observations and flag possible corporate-action jumps.
# Select last 10 sessions and construct a clean EQ-only price panel.
sessions = list(session_audit['date'].tail(WINDOW))
eq = raw.loc[raw['date'].isin(sessions) & raw['SctySrs'].eq('EQ')].copy()
eq['isin'] = eq['ISIN'].astype('string').str.strip()
eq['symbol'] = eq['TckrSymb'].astype('string').str.strip()
for src, dst in [('ClsPric','close'),('PrvsClsgPric','prev_close'),
                 ('TtlTrfVal','traded_value'),('TtlTradgVol','volume')]:
    eq[dst] = pd.to_numeric(eq[src], errors='coerce')
if eq.duplicated(['date','isin']).any():
    dup = eq.loc[eq.duplicated(['date','isin'], keep=False), ['date','isin','symbol']].head()
    display(dup)
    raise ValueError('Duplicate EQ ISIN/date observations; investigate before screening.')
# Basic record validation; flag suspected unadjusted price discontinuities.
eq['bad'] = (eq['isin'].isna() | (eq['isin'].str.len() != 12) |
             eq['symbol'].isna() | ~np.isfinite(eq['close']) | (eq['close'] <= 0) |
             ~np.isfinite(eq['prev_close']) | (eq['prev_close'] <= 0) |
             ~np.isfinite(eq['traded_value']) | (eq['traded_value'] < 0) |
             ~np.isfinite(eq['volume']) | (eq['volume'] <= 0))
eq['daily_move'] = eq['close'] / eq['prev_close'] - 1
eq['suspect_jump'] = eq['daily_move'].abs() > MAX_ABS_DAILY_MOVE
# Check previous-close continuity against prior session close for each ISIN;
# small deviations may reflect adjustments or corrections, so only flag > 10%.
eq = eq.sort_values(['isin','date'])
eq['prior_observed_close'] = eq.groupby('isin')['close'].shift()
eq['prior_observed_date'] = eq.groupby('isin')['date'].shift()
eq['sid'] = eq['date'].map({d:i for i,d in enumerate(sessions)})
eq['prior_sid'] = eq.groupby('isin')['sid'].shift()
consecutive = eq['sid'].sub(eq['prior_sid']).eq(1)
eq['suspect_discontinuity'] = (consecutive &
    ((eq['prev_close'] / eq['prior_observed_close'] - 1).abs() > 0.10))
eq['bad'] = eq['bad'] | eq['suspect_jump'] | eq['suspect_discontinuity']
print('EQ rows:',len(eq),'| flagged rows:',int(eq['bad'].sum()))
print('Screen dates:',sessions[0].date(),'to',sessions[-1].date())

EQ rows: 27266 | flagged rows: 12
Screen dates: 2026-10-27 to 2026-11-09


In [4]:
# CELL 4 — Rank stocks by positive OLS slope of log closes and R² > threshold.
# WHY: select strong, relatively smooth short-term momentum with sufficient liquidity.
# OLS on log(close) for every ISIN with exactly 10 clean consecutive observations.
x = np.arange(WINDOW, dtype=float)
xc = x - x.mean()
sxx = np.dot(xc, xc)
records = []
exclusions = {'incomplete_or_flagged': 0, 'nonpositive_slope': 0, 'low_r2': 0, 'low_liquidity': 0}
for isin, g in eq.groupby('isin', sort=False):
    g = g.sort_values('date')
    if len(g) != WINDOW or g['sid'].tolist() != list(range(WINDOW)) or g['bad'].any():
        exclusions['incomplete_or_flagged'] += 1
        continue
    y = np.log(g['close'].to_numpy(dtype=float))
    yc = y - y.mean()
    slope = float(np.dot(xc, yc) / sxx)
    if slope <= 0:
        exclusions['nonpositive_slope'] += 1
        continue
    resid = yc - slope * xc
    sst = float(np.dot(yc, yc))
    r2 = float(1 - np.dot(resid, resid) / sst) if sst > 0 else 0.0
    if not r2 > MIN_R2:
        exclusions['low_r2'] += 1
        continue
    avg_value = float(g['traded_value'].mean())
    if avg_value < MIN_AVG_TRADED_VALUE:
        exclusions['low_liquidity'] += 1
        continue
    last = g.iloc[-1]
    records.append({'symbol':last['symbol'], 'isin':isin, 'close':float(last['close']),
                    'slope':slope, 'r2':r2, 'avg_traded_value':avg_value,
                    'avg_volume':float(g['volume'].mean()),
                    'ten_session_return_pct':100*(float(last['close'])/float(g.iloc[0]['close'])-1),
                    'approx_daily_growth_pct':100*np.expm1(slope)})
ranked = pd.DataFrame.from_records(records, columns=[
    'symbol','isin','close','slope','r2','avg_traded_value','avg_volume',
    'ten_session_return_pct','approx_daily_growth_pct'])
ranked = ranked.sort_values(['slope','r2'], ascending=[False,False]).reset_index(drop=True)
ranked.index = ranked.index + 1
ranked.index.name = 'rank'
print('Qualified:',len(ranked),'| Exclusions:',exclusions)
display(ranked.head(30).round({'close':2,'slope':5,'r2':3,'avg_traded_value':0,
                                'avg_volume':0,'ten_session_return_pct':2,'approx_daily_growth_pct':2}))

Qualified: 327 | Exclusions: {'incomplete_or_flagged': 144, 'nonpositive_slope': 577, 'low_r2': 1527, 'low_liquidity': 212}


,symbol,isin,close,slope,r2,avg_traded_value,avg_volume,ten_session_return_pct,approx_daily_growth_pct
rank,,,,,,,,,
1,RAMKY,INE874I01013,401.30,0.04148,0.913,4.704812e+08,1444950.0,56.91,4.24
2,FAZE3Q,INE963C01033,540.45,0.04023,0.840,1.608760e+08,306899.0,30.86,4.10
3,SUPREMEINF,INE550H01011,71.21,0.03871,0.843,1.306388e+07,208641.0,29.36,3.95
4,RUSHIL,INE573K01025,20.13,0.03720,0.885,2.373221e+07,1314843.0,35.56,3.79
5,GYFTR,INE724A01017,209.65,0.03640,0.813,1.748182e+08,870329.0,27.98,3.71
6,HEGAM,INE545A01024,289.18,0.03597,0.857,1.486099e+09,5400966.0,34.33,3.66
7,MENONBE,INE071D01033,342.95,0.03484,0.823,3.261809e+08,1081080.0,25.35,3.55
8,TRIGYN,INE948A01012,60.25,0.03288,0.841,1.262997e+07,209438.0,28.88,3.34
9,AGIIL,INE976R01033,296.89,0.03154,0.915,1.155703e+09,4484277.0,27.42,3.20


In [5]:
# CELL 5 — Broad-market benchmark for the same 10 sessions.
# WHY: distinguish stock-specific momentum from an overall market upswing.
# PureMomentum — Broad Market Benchmark

valid = eq.loc[~eq['bad']].sort_values(['isin', 'date'])

returns = (
    valid.groupby('isin')
    .agg(
        sessions=('close', 'count'),
        first_close=('close', 'first'),
        last_close=('close', 'last')
    )
)

returns = returns[returns['sessions'] == WINDOW].copy()

returns['return_pct'] = (
    (returns['last_close'] / returns['first_close'] - 1) * 100
)

r = returns['return_pct']

print(f"Stocks analysed       : {len(r)}")
print(f"Median 10-session return: {r.median():.2f}%")
print(f"Mean 10-session return  : {r.mean():.2f}%")
print(f"Stocks with gains       : {(r > 0).mean()*100:.1f}%")
print(f"Stocks with gains >10%  : {(r > 10).mean()*100:.1f}%")

Stocks analysed       : 2643
Median 10-session return: 4.53%
Mean 10-session return  : 4.72%
Stocks with gains       : 76.3%
Stocks with gains >10%  : 18.5%


In [6]:
# CELL 6 — Inspect the highest-ranked candidates.
# WHY: provide an auditable view before portfolio and trend-age checks.
# PureMomentum — Top 10 candidates (before the last-day crack filter)

display(ranked.head(10).round({
    'close': 2,
    'slope': 5,
    'r2': 3,
    'avg_traded_value': 0,
    'avg_volume': 0,
    'ten_session_return_pct': 2,
    'approx_daily_growth_pct': 2
}))

,symbol,isin,close,slope,r2,avg_traded_value,avg_volume,ten_session_return_pct,approx_daily_growth_pct
rank,,,,,,,,,
1,RAMKY,INE874I01013,401.30,0.04148,0.913,4.704812e+08,1444950.0,56.91,4.24
2,FAZE3Q,INE963C01033,540.45,0.04023,0.840,1.608760e+08,306899.0,30.86,4.10
3,SUPREMEINF,INE550H01011,71.21,0.03871,0.843,1.306388e+07,208641.0,29.36,3.95
4,RUSHIL,INE573K01025,20.13,0.03720,0.885,2.373221e+07,1314843.0,35.56,3.79
5,GYFTR,INE724A01017,209.65,0.03640,0.813,1.748182e+08,870329.0,27.98,3.71
6,HEGAM,INE545A01024,289.18,0.03597,0.857,1.486099e+09,5400966.0,34.33,3.66
7,MENONBE,INE071D01033,342.95,0.03484,0.823,3.261809e+08,1081080.0,25.35,3.55
8,TRIGYN,INE948A01012,60.25,0.03288,0.841,1.262997e+07,209438.0,28.88,3.34
9,AGIIL,INE976R01033,296.89,0.03154,0.915,1.155703e+09,4484277.0,27.42,3.20


In [10]:
# CELL 7 — SELL/HOLD placeholder (not yet operational).
# WHY: the existing tradebook will eventually determine which positions must be sold
# or held before the BUY module chooses new names. No sell decision is fabricated.
# Future input: Zerodha EQ tradebook CSV with complete PureMomentum trade history.
# Future logic: reconstruct open lots, use Monday close and top-10 ranks, then
# apply the agreed first-week loss / 3% gain / top-10 hold and second-week exit rules.
# For now, explicitly declare no tracked PureMomentum holdings. Do NOT use this
# placeholder for live orders while existing PM positions remain open.
sell_hold = pd.DataFrame(columns=['symbol', 'isin', 'quantity', 'decision', 'reason'])
held_symbols = set()  # TODO: replace with reconstructed OPEN positions, SELL or HOLD
print('SELL/HOLD: PLACEHOLDER ONLY — no Zerodha tradebook processed.')


SELL/HOLD: PLACEHOLDER ONLY — no Zerodha tradebook processed.


In [7]:
# CELL 7 — Reject last-day price declines among the ORIGINAL top 10.
# WHY: a decline from the penultimate to the latest completed NSE session may
# signal a crack in otherwise strong short-term momentum. This is a deliberate
# experimental exclusion rule, not proof that the trend has reversed.
# Compare actual consecutive NSE closing prices, NOT Yahoo data or intraday LTP.
# Keep original ranking and examine only these ten; do not replace exclusions
# with ranks 11 onward. Exactly equal closes are NOT treated as a decline.
original_top10 = ranked.head(10).copy()
latest_two = sessions[-2:]
recent_closes = (eq.loc[eq['date'].isin(latest_two), ['isin', 'date', 'close']]
                 .pivot(index='isin', columns='date', values='close'))
crack_rows = []
for rank, candidate in original_top10.iterrows():
    isin = candidate['isin']
    previous = recent_closes.at[isin, latest_two[0]] if isin in recent_closes.index else np.nan
    latest = recent_closes.at[isin, latest_two[1]] if isin in recent_closes.index else np.nan
    verified = bool(np.isfinite(previous) and np.isfinite(latest) and previous > 0 and latest > 0)
    declined = bool(verified and latest < previous)
    crack_rows.append({'rank': int(rank), 'symbol': candidate['symbol'], 'isin': isin,
                       'previous_close': previous, 'latest_close': latest,
                       'last_day_return_pct': 100 * (latest / previous - 1) if verified else np.nan,
                       'last_day_drop': declined,
                       'status': 'DROP' if declined else ('PASS' if verified else 'UNVERIFIED')})
crack_check = pd.DataFrame(crack_rows)
crack_pass_ranks = set(crack_check.loc[crack_check['status'].eq('PASS'), 'rank'])
crack_survivors = original_top10.loc[original_top10.index.isin(crack_pass_ranks)].copy()
print(f'Last-day check: {len(crack_survivors)} of original 10 survive; '
      f'{int(crack_check.last_day_drop.sum())} declined.')
display(crack_check)


Last-day check: 5 of original 10 survive; 5 declined.


,rank,symbol,isin,previous_close,latest_close,last_day_return_pct,last_day_drop,status
0,1,RAMKY,INE874I01013,392.00,401.30,2.372449,False,PASS
1,2,FAZE3Q,INE963C01033,550.00,540.45,-1.736364,True,DROP
2,3,SUPREMEINF,INE550H01011,70.71,71.21,0.707114,False,PASS
3,4,RUSHIL,INE573K01025,19.30,20.13,4.300518,False,PASS
4,5,GYFTR,INE724A01017,210.86,209.65,-0.573840,True,DROP
5,6,HEGAM,INE545A01024,293.87,289.18,-1.595944,True,DROP
6,7,MENONBE,INE071D01033,339.00,342.95,1.165192,False,PASS
7,8,TRIGYN,INE948A01012,61.35,60.25,-1.792991,True,DROP
8,9,AGIIL,INE976R01033,300.93,296.89,-1.342505,True,DROP
9,10,NITCO,INE858F01012,84.65,88.03,3.992912,False,PASS


In [8]:
# CELL 9 — Examine extended momentum for survivors of the original top 10.
# WHY: the 12-session NSE input cannot reveal whether a short-term trend is mature.
# Fetch the last 40 daily closes from Yahoo Finance (not 40 full NSE bhavcopies).
# Operational definition: "7-week upswing" = positive OLS log-price slope and
# R² > 0.80 over the last 35 trading sessions. This is a TREND-STRENGTH PROXY,
# not proof of 35 consecutive rising sessions or the true age of the trend.
# Preserve the original 10-session rankings; exclude flagged candidates from BUY.
# Fail closed for unavailable / mismatched histories: do not issue a BUY for them.
import sys, subprocess
try:
    import yfinance as yf
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'yfinance'])
    import yfinance as yf

trend_rows = []
for rank, candidate in crack_survivors.iterrows():
    symbol = str(candidate['symbol'])
    record = {'rank': int(rank), 'symbol': symbol, 'status': 'HISTORY_UNAVAILABLE',
              'history_sessions': 0, 'trend_slope_35': np.nan,
              'trend_r2_35': np.nan, 'extended_7w': False}
    try:
        hist = yf.Ticker(symbol + '.NS').history(period='6mo', interval='1d', auto_adjust=False)
        hist = hist.loc[hist.index.date <= sessions[-1].date()].sort_index()
        prices = pd.to_numeric(hist['Close'], errors='coerce').dropna().tail(TREND_LOOKBACK)
        record['history_sessions'] = len(prices)
        if len(prices) >= TREND_LOOKBACK and (prices > 0).all():
            # Cross-check Monday's close to guard against symbol mismatches or stale data.
            if abs(float(prices.iloc[-1]) / float(candidate['close']) - 1) > 0.005:
                record['status'] = 'CLOSE_MISMATCH'
            elif prices.index[-1].date() != sessions[-1].date():
                record['status'] = 'STALE_HISTORY'
            else:
                y = np.log(prices.tail(EXTENDED_TREND_SESSIONS).to_numpy(dtype=float))
                x = np.arange(len(y), dtype=float)
                xc, yc = x - x.mean(), y - y.mean()
                slope = float(np.dot(xc, yc) / np.dot(xc, xc))
                resid = yc - slope * xc
                sst = float(np.dot(yc, yc))
                r2 = float(1 - np.dot(resid, resid) / sst) if sst > 0 else 0.0
                record.update(trend_slope_35=slope, trend_r2_35=r2,
                              extended_7w=(slope > 0 and r2 > TREND_MIN_R2),
                              status='EXTENDED' if slope > 0 and r2 > TREND_MIN_R2 else 'PASS')
        else:
            record['status'] = 'INSUFFICIENT_HISTORY'
    except Exception as exc:
        record['status'] = 'FETCH_ERROR'
        print(f'{symbol}: historical lookup failed ({type(exc).__name__}: {exc})')
    trend_rows.append(record)
trend_check = pd.DataFrame(trend_rows, columns=['rank', 'symbol', 'status', 'history_sessions',
                                              'trend_slope_35', 'trend_r2_35', 'extended_7w'])
print('Seven-week trend screen (last-day survivors of original top 10):')
display(trend_check)
# Only verified, non-extended candidates among the last-day survivors of the ORIGINAL top 10 may be bought.
buy_eligible = ranked.loc[ranked.index.isin(trend_check.loc[trend_check.status.eq('PASS'), 'rank'])].copy()
print(f'Eligible among top 10: {len(buy_eligible)}; extended/unverified candidates excluded.')


Seven-week trend screen (last-day survivors of original top 10):


,rank,symbol,status,history_sessions,trend_slope_35,trend_r2_35,extended_7w
0,1,RAMKY,PASS,40,0.002457,0.045015,False
1,3,SUPREMEINF,PASS,40,-0.007467,0.190039,False
2,4,RUSHIL,PASS,40,0.001523,0.036787,False
3,7,MENONBE,PASS,40,0.000477,0.003919,False
4,10,NITCO,PASS,40,-0.005492,0.369234,False


Eligible among top 10: 5; extended/unverified candidates excluded.


In [11]:
# CELL 10 — Produce a discretionary shortlist of up to THREE candidates.
# WHY: show the best three eligible stocks after SELL, last-day decline, and extended-trend checks;
# the user chooses which (if any) to trade. No automatic purchase occurs.
# Pricing uses Monday's LAST AVAILABLE NSE CLOSE, not a live Zerodha LTP.
# Calculate a proposed LIMIT 0.5% above that close, rounded UP to a tick.
# Indicative quantity is sized to at most half the weekly ₹10,000 budget PER
# SELECTED STOCK; do not add all three illustrative costs together as a plan.
# Choosing two stocks at these quantities respects the weekly ₹10,000 ceiling.
allocation = WEEKLY_BUDGET * MAX_POSITION_PCT
picks = []
for rank, row in buy_eligible.iterrows():
    if str(row["symbol"]) in EXCLUDE_SYMBOLS or str(row["symbol"]) in held_symbols:
        continue
    buy_limit = round(np.ceil((float(row["close"]) * (1 + BUY_PREMIUM) - 1e-9) / TICK_SIZE) * TICK_SIZE, 2)
    qty = int(allocation // buy_limit)
    if qty < 1:
        continue
    picks.append({'rank':rank, 'symbol':row['symbol'], 'isin':row['isin'],
                  'reference_close':row['close'], 'buy_limit':buy_limit, 'indicative_qty':qty,
                  'estimated_cost':round(qty*buy_limit, 2), 'slope':row['slope'], 'r2':row['r2']})
    if len(picks) == SHORTLIST_SIZE:
        break
orders = pd.DataFrame(picks)
print(f'Discretionary shortlist: {len(orders)} of {SHORTLIST_SIZE} candidates | '
      f'Weekly purchase ceiling: ₹{WEEKLY_BUDGET:,.2f} | '
      f'Indicative cap per selected stock: ₹{allocation:,.2f}')
if len(orders) < SHORTLIST_SIZE:
    print('WARNING: Fewer than three affordable qualifying candidates.')
display(orders.round(4))
print('Choose up to TWO stocks at the indicative quantities; do not buy all three at these sizes.')
print('IMPORTANT: reference_close is the latest NSE close, NOT live LTP. '
      'A limit is a maximum price, not a guaranteed execution.')


Discretionary shortlist: 3 of 3 candidates | Weekly purchase ceiling: ₹10,000.00 | Indicative cap per selected stock: ₹5,000.00


,rank,symbol,isin,reference_close,buy_limit,indicative_qty,estimated_cost,slope,r2
0,1,RAMKY,INE874I01013,401.30,403.35,12,4840.2,0.0415,0.9127
1,3,SUPREMEINF,INE550H01011,71.21,71.60,69,4940.4,0.0387,0.8435
2,4,RUSHIL,INE573K01025,20.13,20.25,246,4981.5,0.0372,0.8854


Choose up to TWO stocks at the indicative quantities; do not buy all three at these sizes.
IMPORTANT: reference_close is the latest NSE close, NOT live LTP. A limit is a maximum price, not a guaranteed execution.


In [12]:
# CELL 11 — Export screening, diagnostics, and indicative orders.
# WHY: preserve the evidence and decisions for weekly audit and comparison.
# Save outputs and download a single results bundle.
from datetime import datetime
stamp = sessions[-1].strftime('%Y%m%d')
paths = {
    f'PureMomentum_ranked_{stamp}.csv': ranked.reset_index().to_csv(index=False),
    f'PureMomentum_orders_{stamp}.csv': orders.to_csv(index=False),
    f'PureMomentum_trend_check_{stamp}.csv': trend_check.to_csv(index=False),
    f'FreshMomentum_last_day_check_{stamp}.csv': crack_check.to_csv(index=False),
    f'PureMomentum_sell_hold_PLACEHOLDER_{stamp}.csv': sell_hold.to_csv(index=False),
    f'PureMomentum_session_audit_{stamp}.csv': session_audit.to_csv(index=False),
}
notes = (f'Latest session: {sessions[-1].date()}\nFormation window: {WINDOW} sessions\n'
         f'Slope > 0; R2 > {MIN_R2}; mean TtlTrfVal >= INR {MIN_AVG_TRADED_VALUE:,}\n'
         f'Candidates: {len(ranked)}; exclusion counts: {exclusions}\n'
         'Unadjusted closes; suspected discontinuities flagged, not corporate-action-adjusted.\n'
         f'Discretionary shortlist: up to {SHORTLIST_SIZE}; select up to two at indicated quantities.\n'
         f'Buy limit: {BUY_PREMIUM:.1%} above latest NSE close, NOT live LTP (tick-rounded up); size at limit.\n'
         'Last-day filter: exclude latest close below previous close within original top 10; no replenishment.\n'
         f'Trend-age proxy: {EXTENDED_TREND_SESSIONS}-session positive log slope and R2 > {TREND_MIN_R2}; top 10 only.\n'
         'Unavailable history excluded; SELL module is a placeholder, not operational.\n'
         'Weekly exit/holding rules require separately recorded actual fills and subsequent prices.\n')
paths[f'PureMomentum_notes_{stamp}.txt'] = notes
#out = f'PureMomentum_results_{stamp}.zip'
out = f'NSE12_{stamp}_Output.zip'



with zipfile.ZipFile(out, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for filename, contents in paths.items():
        z.writestr(filename, contents)
print('Saved:',out)
files.download(out)

Saved: NSE12_20261109_Output.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Interpretation and limitations

- This is a **Monday-close screen**, not a guarantee of Tuesday availability or execution price.
- The latest **10 of the 12** supplied sessions are used. The extra two sessions provide acquisition redundancy, not additional regression observations.
- `TtlTrfVal` is the **actual daily traded value in rupees**, averaged over the ten sessions.
- A 40% daily move or a 10% previous-close discontinuity is conservatively excluded as suspicious, **not proven** to be a corporate action. Other splits, bonuses or adjustments may escape these checks. Inspect unusual names before trading.
- The notebook does **not** automatically enforce the one-week/second-week exit rules: those require a persistent position ledger with actual Tuesday fills and later prices.
- No transaction charges, taxes or slippage are included in the indicative allocation.
- Extended-history checks use Yahoo Finance, which can fail or differ from official NSE data; unmatched/unavailable histories are excluded from BUY.
- The 35-session OLS proxy does not establish that a stock rose continuously for seven weeks. This filter has **not** been backtested.
- SELL/HOLD remains a dummy cell; it must be implemented before relying on MNA to manage live PureMomentum holdings.
